### Define function for Task 2
copying the function that I wrote for task 1 and adding analysis of where the cars are moving and count the cars moving to city center 

In [1]:
# install OpenCV python library 
!pip install opencv-python

  Using cached opencv_python-4.10.0.84-cp37-abi3-macosx_11_0_arm64.whl.metadata (20 kB)
Using cached opencv_python-4.10.0.84-cp37-abi3-macosx_11_0_arm64.whl (54.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 24.9 MB/s eta 0:00:00a 0:00:01


In [2]:
# import library to analyze and play video 
import cv2

# define the function for task two
# funcion will:
# 	1. take video file name 
# 	2. analyze video (frame differencing, background substruction)
#   3. count how many cars is moving in that direction 
# 	4. return video feed with added effect on moving cars (green rectangles)
#   5. return amount of cars 


# most of the code below is from task 1 
def countCars(fileName):
  # TAKE VIDEO FILE NAME 
  video = cv2.VideoCapture(fileName)
  if (video.isOpened()== False): 
    print("Error opening video file")
  
  # ANALYZE 
  # for analyzis function will need 2 frames, current and previous to compare the difference between both to track changes as movement 
  backgroundFrame = None


  # point of intrest, that will coun number of cars passing that point, point is to be set to count cars moving to city center
  xIntrest = 490
  yIntrest = 390

  # variables for the car counter 
  carDetected = False
  wasCarDetectedLastFrame = False
  framesOfFalse = 0 # to make sure same car is not counted more then once 
  numOfCars = 0

  # iterate over every single frame of the video 
  while True:
    # get status, and frame from video object 
    check, frame = video.read()
    # if eof or other issues exit while loop 
    if not check:
      cv2.destroyAllWindows()
      cv2.waitKey(1)
      return(numOfCars)
    
    # for frame differencing and background substruction: change to gray, blur, 
    # change frame to gray 
    gray_frame=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

    # blur the frame 
    blur_frame=cv2.GaussianBlur(gray_frame,(25,25),0)

    #first iteration
    if backgroundFrame is None:
      backgroundFrame = blur_frame
   
    # substract background - find the difference in the frames
    delta_frame=cv2.absdiff(backgroundFrame,blur_frame)

    # delta frame results in black where is no movement, and gray where there is movement 
    # convert grayscale image into binary: black and white frame, where white are the objects that are moving 
    # analyzing shade of gray, is it whith in a threshold pixels returns true and is white, other wise its black 
    # result is binary black and white frame 
    threshold_frame=cv2.threshold(delta_frame,6,255, cv2.THRESH_BINARY)[1]

    # create contours of binary threshold frame 
    (contours,_)=cv2.findContours(threshold_frame,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    # loop over contours 
    for c in contours:
          # only track cars so skip all contours smaller then 3000 
          if cv2.contourArea(c) < 3000:
              continue
          # create and draw ractangle based on the current contour thas is bigger than 3000 
          (x, y, w, h)=cv2.boundingRect(c)
          cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 1)
          # check if there is a rectangle in the interest point to count cars 
          if(x <= xIntrest and x+w >= xIntrest and y <= yIntrest and y+h >= yIntrest):
            carDetected = True

    # car counter 
    framesOfFalse = framesOfFalse + 1 
    if wasCarDetectedLastFrame == False and carDetected == True and framesOfFalse > 30:
      # I had issue where car has windows and filter black to white is missreading pixel differencing, because of that countour is being lost in the are for couple frames and same car 
      # is counted more than once
      # to address that issue I have a connept that before counting another car there have to be at least 30 frames of car not being detected
      # that should prevent same car being counted twice, and space between 2 cars should be I assume at 30 frames 
      numOfCars = numOfCars + 1
      framesOfFalse = 0
    # save info from this frame 
    wasCarDetectedLastFrame = carDetected
    # set the value for next frame 
    carDetected = False
    
    # VIDEO FEED
    cv2.imshow('Detector', frame)

    # To exit quit viedo early press q 
    if cv2.waitKey(1) & 0xFF == ord("q"):
      cv2.destroyAllWindows()
      cv2.waitKey(1)
      return(numOfCars)


# funcion to calculate time of the video, to be able to make a table with how many cars pass per minute 
def timeOfVideo(fileName):
  # I COPIED CODE BELOW FROM https://stackoverflow.com/questions/49048111/how-to-get-the-duration-of-video-using-opencv AND ADDAPTED THE VARIABLES TO MY CODE 
  # accessed JUL 15 2024, author: ivan_pozdeev Mar 1, 2018, Title: python - How to get the duration of video using OpenCV

  # COPIED CODE STARTS HERE
  video = cv2.VideoCapture(fileName)
  fps = video.get(cv2.CAP_PROP_FPS) 
  frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
  duration = frame_count/fps
  # COPIED CODE ENDS HERE

  return(duration)



Now I have 2 functions:
- function returning number of cars on the video
- function returning duration of the video in seconds 

Next step is to call those functions on both videos and make a table with data.<br>
To calculate cars per minute, I made a formula:<br>
number of cars / durations of video * 60 = cars per minute 

In [3]:
# Funciotn takes amount of cars, senocnds of video and returns cars per minute 
# import math for ceil funciton 
import math 
def carsPerMinute(numOfCars, seconds):
    return math.ceil(numOfCars / seconds * 60)

In [4]:
# Number of cars video 1 
numOfCarsVideo1 = countCars('Exercise1_Files/Traffic_Laramie_1.mp4')
print("Number of cars: ", numOfCarsVideo1)

# Duration of video 1 
durationOfVideo1 = timeOfVideo('Exercise1_Files/Traffic_Laramie_1.mp4')
print("Duration of video: ", durationOfVideo1)

Number of cars:  6
Duration of video:  177.92


In [5]:
# Number of cars video 2
numOfCarsVideo2 = countCars('Exercise1_Files/Traffic_Laramie_2.mp4')
print("Number of cars: ", numOfCarsVideo2)

# Duration of video 2
durationOfVideo2 = timeOfVideo('Exercise1_Files/Traffic_Laramie_2.mp4')
print("Duration of video: ", durationOfVideo2)

Number of cars:  4
Duration of video:  105.68


### Creating a table
to do so I decided to use tabulate library 
and used documentation to learn how to make a table https://pypi.org/project/tabulate/ accessed Jul/21/24


In [6]:
!pip install tabulate

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)


In [7]:
import tabulate
print(
    tabulate.tabulate(
      [
        ["Traffic_Laramie_1.mp4", numOfCarsVideo1, carsPerMinute(numOfCarsVideo1, durationOfVideo1)], 
        ["Traffic_Laramie_2.mp4", numOfCarsVideo2, carsPerMinute(numOfCarsVideo2, durationOfVideo2)]
      ],
      headers=["", "Total number of cars", "Cars per minute"],
      tablefmt="simple_grid"
    )
)


┌───────────────────────┬────────────────────────┬───────────────────┐
│                       │   Total number of cars │   Cars per minute │
├───────────────────────┼────────────────────────┼───────────────────┤
│ Traffic_Laramie_1.mp4 │                      6 │                 3 │
├───────────────────────┼────────────────────────┼───────────────────┤
│ Traffic_Laramie_2.mp4 │                      4 │                 3 │
└───────────────────────┴────────────────────────┴───────────────────┘
